# Preparation carbon stocks data (figure m06)

In [ ]:
# Libraries
import xarray as xr
import pandas as pd

In [ ]:
# Directories
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'
dir07 = '../paper_ecosystem_carbon/output/07_eval_prep/'
dir_out = '../paper_ecosystem_carbon/output/zenodo/'

---

### Prepare data

In [ ]:
# Directory with processed blue data
dir07_bluep = f'{dir07}blue_processed/'

# Define blue run strings
run_chansis = 'run_20260515_default_1700-2024'
run_cspatial = 'run_20260515_cspatial_1700-2024'
run_cdgvm = 'run_20260515_cdgvm_1700-2024'

In [ ]:
# Get underlying cveg and csoil data
ds_cveg = xr.open_zarr(f'{dir02}ds_cveg_biom_r001.zarr')
ds_csoil = xr.open_zarr(f'{dir02}ds_csoil_000_100_r001.zarr')

# Get grid cell area
ds_area = xr.open_zarr(f'{dir07}ds_prep_area_r001.zarr')

# Get prepared trendy data
ds_trendy_ysum = xr.open_dataset(f'{dir07}ds_trendy_ysum.nc')

In [ ]:
# Create datasets with yearly global values of eluc and cpools data
def get_blue_data_cveg_csoil(dir_in, blue_run,
                   chunk_time=50, chunk_lat=-1, chunk_lon=-1):
    """Returns prepared blue data as lazy dataset."""
    ds = xr.open_dataset(f'{dir_in}{blue_run}_cveg_csoil.nc', 
                           chunks='auto') \
        .chunk(chunks=dict(time=chunk_time, lat=chunk_lat, lon=chunk_lon))
    return ds


def prep_ds_ysum(dir_in, blue_run):
    """Returns yearly global values."""
    
    ds = get_blue_data_cveg_csoil(dir_in, blue_run)
    ds_ysum = (ds[['cveg', 'csoil']]* ds.cell_area).sum(['lat', 'lon']) * 1E-9
    
    return ds_ysum.compute()


ds_chansis_ysum = prep_ds_ysum(dir07_bluep, run_chansis)
ds_cspatial_ysum = prep_ds_ysum(dir07_bluep, run_cspatial)
ds_cdgvm_ysum = prep_ds_ysum(dir07_bluep, run_cdgvm)

In [ ]:
# Trendy data for year 2022
ds_trendy_2022 = ds_trendy_ysum.sel(time=2022)

In [ ]:
# Calculate global carbon values of underlying cveg and csoil data as reference data
ds_ref = xr.merge([ds_cveg, ds_csoil], join='exact')
ds_ref_sum = (ds_ref * ds_area.area_ha).sum(['lat', 'lon']).compute() * 1E-9

v_ref_cveg = ds_ref_sum.cveg_biom.item()
v_ref_csoil = ds_ref_sum.csoil_000_100.item()

In [ ]:
# Values from literature
dict_lit = {
    'spawn_cveg': 409,
    'erb_cveg': 450,
    'walker_cveg': 441.2,
    'sanderman_csoil': 1998.1, 
    'batjes_csoil': 1408,
    'tifafi-soilgrids_csoil': 3421,
    'tifafi-hwsd_csoil': 2439
}

---

### Create and export dataframe with carbon pool data

In [ ]:
# Create dataframe for export
df_out = pd.DataFrame(
    columns=['source', 'source_type', 'carbon_type', 'value_pgc']
)

# Add blue values
dict_ds_blue_ysum = {
    'chansis': ds_chansis_ysum,
    'cspatial': ds_cspatial_ysum,
    'cdgvm': ds_cdgvm_ysum
}

def add_blue_run(ds, name, year, var):
    return [f'blue_{name}_{year}', 
            'blue', 
            var, 
            round(ds.sel(time=year)[var].item())]


for year in [1800, 2022]:
    for var in ['cveg', 'csoil']:
        for name, ds in dict_ds_blue_ysum.items():
            df_out.loc[len(df_out)] = add_blue_run(ds, name, year, var)


# Add trendy data
for var in ds_trendy_2022.data_vars:
    source, carbon_type = var.rsplit('_', 1)
    value = round(ds_trendy_2022[var].item())

    df_out.loc[len(df_out)] = (f'{source}_2022', 
                               'trendyv12', 
                               carbon_type, 
                               value)


# Add reference data
df_out.loc[len(df_out)] = ('reference', 'other', 'cveg', round(v_ref_cveg))
df_out.loc[len(df_out)] = ('reference', 'other', 'csoil', round(v_ref_csoil))


# Add literature data
for i in dict_lit.keys():
    s, t = i.split('_')[0], i.split('_')[1]
    df_out.loc[len(df_out)] = (s, 'literature', t, round(dict_lit[i]))

# Set values to integer
df_out['value_pgc'] = df_out['value_pgc'].astype('int64')

# Export
df_out.to_csv(f'{dir_out}ecosystem_carbon_data_figure_m06.csv', index=False)

---

### Check

In [ ]:
df_out